In [11]:
import os
import json
import asyncio
import nest_asyncio
nest_asyncio.apply()

from anthropic import AsyncAnthropic

client = AsyncAnthropic(api_key="<Enter your APIs>")  # ⚠️ Move this to an env var!

# ==========================================
# MOCK MCP SERVER SYSTEM DATA & TOOLS
# ==========================================
HR_POLICY_DATABASE = {
    "leave_policy": "Employees receive 25 days of paid annual leave. Paternity/Maternity leave is 12 weeks fully paid. Casual leave is capped at 5 days per year.",
    "expense_caps": "Daily meals are capped at $75/day during international travel. Accommodation is capped at $250/night for Tier 1 cities."
}

TRAVEL_LOG_DATABASE = {
    "flight_availability": "SFO to LHR: BA286 (Available, $1200), UA901 (Available, $1450). BLR to SFO: EK565 (Available, $1800).",
    "active_bookings": "EmpID 9804 has an active booking to Paris on July 14th via Air France."
}

# ==========================================
# AGENT MODULE 1: THE HR SPECIALIST (OPUS)
# ==========================================
async def run_hr_agent(instruction: str) -> str:
    print(" -> [Opus] Processing HR Request...")
    response = await client.messages.create(
        model="claude-opus-4-6",  # ✅ Updated
        max_tokens=600,
        temperature=0,
        system=f"You are the specialized HR Data Agent. You have direct access to the HR Knowledge base: {json.dumps(HR_POLICY_DATABASE)}. Answer questions strictly based on this data.",
        messages=[{"role": "user", "content": instruction}]
    )
    return response.content[0].text

# ==========================================
# AGENT MODULE 2: THE TRAVEL AGENT (HAIKU)
# ==========================================
async def run_travel_agent(instruction: str) -> str:
    print(" -> [Haiku] Processing Travel Logistics...")
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",  # ✅ Updated
        max_tokens=600,
        temperature=0,
        system=f"You are the rapid Travel Coordinator Agent. You have direct access to the Travel systems: {json.dumps(TRAVEL_LOG_DATABASE)}. Keep responses concise and factual.",
        messages=[{"role": "user", "content": instruction}]
    )
    return response.content[0].text

# ==========================================
# THE CORE ORCHESTRATOR (SONNET)
# ==========================================
async def coordinate_task(user_input: str):
    print(f"[Sonnet Orchestrator] Analyzing master request: \"{user_input}\"")
    
    classification_prompt = f"""
    You are the Central Orchestrator Agent powered by Claude Sonnet. 
    Your job is to read the user request and determine if it requires assistance from our specialized downstream sub-agents:
    1. HR Agent (Handles corporate leaves, policies, budgets, and compliance)
    2. Travel Agent (Handles flights, travel dates, availability, and bookings)

    Analyze this request: "{user_input}"
    
    Respond ONLY with a valid JSON object. No markdown, no code fences, no explanation. Just the raw JSON:
    {{
      "requiresHR": true,
      "hrInstruction": "Specific curated prompt to hand to the HR Agent, or empty string",
      "requiresTravel": true,
      "travelInstruction": "Specific curated prompt to hand to the Travel Agent, or empty string"
    }}
    """

    orchestrator_response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=400,
        temperature=0,
        messages=[{"role": "user", "content": classification_prompt}]
    )

    raw_text = orchestrator_response.content[0].text.strip()
    
    # Debug: see exactly what the model returned
    print(f"[Orchestrator Raw Response]:\n{raw_text}\n")

    # Strip markdown code fences if present (```json ... ``` or ``` ... ```)
    if raw_text.startswith("```"):
        raw_text = raw_text.split("```")[1]          # grab content between fences
        if raw_text.startswith("json"):
            raw_text = raw_text[4:]                  # strip the 'json' language tag
        raw_text = raw_text.strip()

    if not raw_text:
        raise ValueError("Orchestrator returned an empty response. Check your API key and prompt.")

    routing_data = json.loads(raw_text)
    
    tasks = {}
    
    if routing_data.get("requiresHR"):
        tasks["hr"] = asyncio.create_task(run_hr_agent(routing_data["hrInstruction"]))
        
    if routing_data.get("requiresTravel"):
        tasks["travel"] = asyncio.create_task(run_travel_agent(routing_data["travelInstruction"]))

    if tasks:
        await asyncio.gather(*tasks.values())
        
    hr_results = tasks["hr"].result() if "hr" in tasks else ""
    travel_results = tasks["travel"].result() if "travel" in tasks else ""

    print("[Sonnet Orchestrator] Compiling individual agent data into master briefing...")
    
    synthesis_prompt = f"""
    Synthesize a final executive answer for the user's inquiry: "{user_input}"
    
    Here is the certified data compiled from our internal sub-agents:
    {f'[HR Agent Insights]: {hr_results}' if hr_results else ''}
    {f'[Travel Agent Insights]: {travel_results}' if travel_results else ''}
    
    Provide a clean, elegant, unified response to the employee. Do not mention the names of the models or the internal routing mechanics.
    """

    final_response = await client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1000,
        temperature=0.3,
        messages=[{"role": "user", "content": synthesis_prompt}]
    )

    print("\n=================== FINAL BRIEFING ===================")
    print(final_response.content[0].text)
    print("======================================================\n")
# ==========================================
# EXECUTION ENTRY POINT
# ==========================================
if __name__ == "__main__":
    complex_employee_query = (
        "I need to check if we can book a flight from SFO to LHR for an upcoming "
        "client meeting, and also check what our daily budget cap is for meals "
        "while traveling so I can submit my pre-clearance forms."
    )
    asyncio.run(coordinate_task(complex_employee_query))


[Sonnet Orchestrator] Analyzing master request: "I need to check if we can book a flight from SFO to LHR for an upcoming client meeting, and also check what our daily budget cap is for meals while traveling so I can submit my pre-clearance forms."
[Orchestrator Raw Response]:
{
  "requiresHR": true,
  "hrInstruction": "The employee is preparing to submit pre-clearance forms for a business trip and needs to know the company's daily budget cap for meals while traveling. Please provide the current meal per diem or daily meal allowance policy for employees on business travel, including any distinctions for domestic vs. international travel if applicable.",
  "requiresTravel": true,
  "travelInstruction": "The employee needs to book a flight from SFO (San Francisco International Airport) to LHR (London Heathrow Airport) for an upcoming client meeting. Please check availability and options for this route, including available dates, flight options, and any relevant booking details or restrict